In [1]:
!pip install kaggle

In [2]:
import os
import json

kaggle_username = "akhi1205"
kaggle_key = "KGAT_0527bc3897e94be0ec5482501d763c71"

kaggle_json = {
    "username": kaggle_username,
    "key": kaggle_key
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_json, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle API configured successfully!")

Kaggle API configured successfully!


In [3]:
!kaggle datasets download -d ggrill/foodseg103

Dataset URL: https://www.kaggle.com/datasets/ggrill/foodseg103
License(s): apache-2.0
100% 1.17G/1.17G [00:15<00:00, 81.1MB/s]



In [4]:
import zipfile

with zipfile.ZipFile("foodseg103.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/foodseg103")

print("Dataset extracted!")

Dataset extracted!


In [19]:
base_path = "/content/foodseg103/FoodSeg103"

image_dir = f"{base_path}/Images/img_dir"
mask_dir = f"{base_path}/Images/ann_dir"

train_txt = f"{base_path}/ImageSets/train.txt"
val_txt   = f"{base_path}/ImageSets/test.txt"

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

shutil.copy(
    "/content/foodseg103/FoodSeg103/category_id.txt",
    "/content/drive/MyDrive/category_id.txt"
)

print("Category file saved to Drive.")

Mounted at /content/drive
Category file saved to Drive.


In [14]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

import torchvision.models as models
import torchvision.transforms as transforms

from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [60]:
class FoodSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, split_file, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        with open(split_file, "r") as f:
            self.image_ids = f.read().splitlines()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]

        # ✅ correct path (NO .jpg added)
        img_path = os.path.join(self.img_dir, img_id)

        # ✅ mask path
        mask_name = img_id.replace(".jpg", ".png")
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        mask = np.array(mask)

        # 🔥 dominant class as label
        unique, counts = np.unique(mask, return_counts=True)
        label = unique[np.argmax(counts)]

        if self.transform:
            image = self.transform(image)

        return image, int(label)

In [54]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [55]:
train_data = FoodSegDataset(image_dir, mask_dir, train_txt, transform)
test_data  = FoodSegDataset(image_dir, mask_dir, val_txt, transform)

In [56]:
import torch

train_data = torch.utils.data.Subset(train_data, range(3000))
test_data  = torch.utils.data.Subset(test_data, range(1000))

In [57]:
model = models.resnet50(pretrained=True)

# remove classifier
model = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [58]:
def extract_features(dataset):
    features = []
    labels = []

    for img, label in dataset:
        img = img.unsqueeze(0)

        with torch.no_grad():
            feature = model(img)

        feature = feature.view(-1).numpy()

        features.append(feature)
        labels.append(label)

    return np.array(features), np.array(labels)

In [62]:
import os
print(os.listdir(image_dir)[:5])

['test', 'train']


In [64]:
image_dir = f"{base_path}/Images/img_dir/train"
mask_dir  = f"{base_path}/Images/ann_dir/train"

test_image_dir = f"{base_path}/Images/img_dir/test"
test_mask_dir  = f"{base_path}/Images/ann_dir/test"

In [65]:
train_data = FoodSegDataset(image_dir, mask_dir, train_txt, transform)

test_data  = FoodSegDataset(test_image_dir, test_mask_dir, val_txt, transform)

In [66]:
train_data = torch.utils.data.Subset(train_data, range(3000))
test_data  = torch.utils.data.Subset(test_data, range(1000))

In [67]:
def extract_features(dataset):
    features = []
    labels = []

    for img, label in dataset:
        img = img.unsqueeze(0)

        with torch.no_grad():
            feature = model(img)

        feature = feature.view(-1).numpy()

        features.append(feature)
        labels.append(label)

    return np.array(features), np.array(labels)

In [68]:
print("Extracting training features...")
X_train, y_train = extract_features(train_data)

print("Extracting testing features...")
X_test, y_test = extract_features(test_data)

Extracting training features...
Extracting testing features...


In [69]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [70]:
clf = svm.SVC(kernel='rbf', C=10)

print("Training SVM...")
clf.fit(X_train, y_train)

Training SVM...


SVC(C=10)

In [71]:
print("Predicting...")
y_pred = clf.predict(X_test)

Predicting...


In [72]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy * 100)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 74.4

Classification Report:

              precision    recall  f1-score   support

           0       0.77      1.00      0.87       730
           3       0.00      0.00      0.00        13
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         3
           8       1.00      0.12      0.22         8
           9       0.00      0.00      0.00         4
          10       0.00      0.00      0.00         8
          13       0.00      0.00      0.00         1
          24       0.00      0.00      0.00         2
          25       0.00      0.00      0.00         1
          30       1.00      0.25      0.40         4
          37       0.00      0.00      0.00         1
          44       0.00      0.00      0.00         1
          46       0.15      0.06      0.09        31
          47       0.29      0.13      0.18        15
          48       0.19      0.23      0.21        26
          49       0.00      0.00      0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
